In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Resolve project root (adjust if needed)
PROJECT_ROOT = Path.cwd()

# If notebook is inside a subfolder, move up
if not (PROJECT_ROOT / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

env_path = PROJECT_ROOT / ".env"
print("Loading env from:", env_path)

load_dotenv(dotenv_path=env_path, override=True)

print("GOOGLE_API_KEY loaded:", bool(os.getenv("GOOGLE_API_KEY")))

#import ADK componets
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.runners import InMemoryRunner
from google.adk.models.google_llm import Gemini
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("Import done")

retry_config = types.HttpRetryOptions(attempts=5, exp_base=7, initial_delay=1,
                                      http_status_codes=[429, 503, 500, 504])



Loading env from: /Users/anujmittal/Desktop/ai_agentic/.venv/.env
GOOGLE_API_KEY loaded: True


/Users/anujmittal/Desktop/ai_agentic/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'


/Users/anujmittal/Desktop/ai_agentic/.venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:252: FutureWarning: You are using a Python version (3.9.6) past its end of life. Google will update google.api_core with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'


/Users/anujmittal/Desktop/ai_agentic/.venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:252: FutureWarning: You are using a Python version (3.9.6) past its end of life. Google will update google.cloud.aiplatform_v1beta1 with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.cloud.aiplatform_v1beta1.
  warnings.warn(message, FutureWarning)


An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'
An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'
An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'
An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'
An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'
An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'
An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'
An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'
An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'


/Users/anujmittal/Desktop/ai_agentic/.venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:252: FutureWarning: You are using a Python version (3.9.6) past its end of life. Google will update google.cloud.aiplatform_v1 with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.cloud.aiplatform_v1.
  warnings.warn(message, FutureWarning)
/Users/anujmittal/Desktop/ai_agentic/.venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:252: FutureWarning: You are using a Python version (3.9.6) past its end of life. Google will update google.cloud.aiplatform.v1.schema.predict.instance_v1 with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.cloud.aiplatform.v1.schema.predict.instance_v1.
  warnings.warn(

Import done


In [2]:
initial_writer_agent = Agent(
    name="InitialWriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Based on the user's prompt, write the first draft of a short story (around 100-150 words).
    Output only the story text, with no introduction or explanation.""",
    output_key="current_story",  # Stores the first draft in the state.
)

print("initial_writer_agent created.")

critic_agent = Agent(
    name="CriticAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a constructive story critic. Review the story provided below.
    Story: {current_story}
    
    Evaluate the story's plot, characters, and pacing.
    - If the story is well-written and complete, you MUST respond with the exact phrase: "APPROVED"
    - Otherwise, provide 2-3 specific, actionable suggestions for improvement.""",
    output_key="critique",  # Stores the feedback in the state.
)

print("✅ critic_agent created.")

initial_writer_agent created.
✅ critic_agent created.


In [3]:
#Write exit loop function
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', 
    indicating the story is finished and no more changes are needed """
    return { "status": "approved", "message": "Story is approved terminate the refinement loop."}
print("exit_loop function")


#Refiner Agent which will critic agent to refine the message
refiner_agent = Agent(
    name="RefinerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a story refiner. You have a story draft and critique.
    
    Story Draft: {current_story}
    Critique: {critique}
    
    Your task is to analyze the critique.
    - IF the critique is EXACTLY "APPROVED", you MUST call the `exit_loop` function and nothing else.
    - OTHERWISE, rewrite the story draft to fully incorporate the feedback from the critique.""",
    output_key="current_story",  # It overwrites the story with the new, refined version.
    tools=[
        FunctionTool(exit_loop)
    ],  # The tool is now correctly initialized with the function reference.
)

print("refiner_agent created.")


# LoopAgent which will orachestrate critic & refiner agent 
story_refinement_loop=LoopAgent(
    name="RefinmentAgent",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=2 # shouldn't be infinite loop
)

#final root agent
root_agent=SequentialAgent(
    name="RootAgent",
    sub_agents=[initial_writer_agent, story_refinement_loop]
)

print("Loop and seuential agents are created")


# Run the root agent
runner=InMemoryRunner(
 agent=root_agent
)
response = await runner.run_debug(
    "Write a story about nice ful moon shine, sparkling stars and clam flowing river."
)

App name mismatch detected. The runner is configured with app name "InMemoryRunner", but the root agent was loaded from "/Users/anujmittal/Desktop/ai_agentic/.venv/lib/python3.9/site-packages/google/adk/agents", which implies app name "agents".


exit_loop function
refiner_agent created.
Loop and seuential agents are created

 ### Created new session: debug_session_id

User > Write a story about nice ful moon shine, sparkling stars and clam flowing river.


CancelledError: 